# Assignment Module 2: Aircraft Classification

The goal of this assignment is to implement neural networks that classify images of 100 aircraft model variants from the [Fine-Grained Visual Classification of Aircraft (FGVC-Aircraft) dataset](https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/). The notebook is divided into two parts:

1. Build and train a custom CNN from scratch, using only PyTorch layers.
2. Fine-tune a PyTorch ResNet-18 pretrained on ImageNet-1K V1.

![](https://raw.githubusercontent.com/CVLAB-Unibo/ipcv-assignment-2/master/fgvc_aircraft_variants.svg)


## Dataset

The dataset is loaded through the official [`torchvision.datasets.FGVCAircraft`](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.FGVCAircraft.html) class with `annotation_level="variant"`. The official split names are used:

- `train` for optimization.
- `val` for model selection and early stopping.
- `test` for final reporting.

All experiments use ImageNet normalization. This keeps preprocessing compatible with ResNet-18 and gives the custom CNN a stable input scale.


## Experiment Roadmap

Part 1 trains the best custom CNN configuration first, then reruns controlled ablations that intentionally remove one helpful component at a time. The ablations are designed to explain why the final architecture uses data augmentation, batch normalization, dropout, and enough channel capacity.

Part 2 starts from ResNet-18 ImageNet-1K V1 weights. The first ResNet experiment reuses the Part 1 training hyperparameters as requested. The second stage changes the hyperparameters for transfer learning: stronger augmentation, lower backbone learning rate, higher classifier-head learning rate, and a frozen-backbone comparison.

Run all training cells to populate the tables and plots. Checkpoints, training histories, and final CSV summaries are written to `outputs/`.


## Global Setup

The controls below are the only switches that should usually be changed. For a quick syntax or pipeline check, set `FAST_DEV_RUN = True`; for the real assignment run, keep it `False` and let all selected experiments train normally.


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms as T

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
HISTORY_DIR = OUTPUT_DIR / "histories"

for directory in (DATA_DIR, OUTPUT_DIR, CHECKPOINT_DIR, HISTORY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
RUN_DOWNLOAD = True
RUN_PART1 = True
RUN_PART2A = True
RUN_PART2B = True
FAST_DEV_RUN = False

NUM_WORKERS = min(4, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Workers: {NUM_WORKERS}")


In [ ]:
def set_seed(seed: int = SEED) -> None:
    """Make runs as reproducible as the active hardware allows."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    image_size: int = 224
    batch_size: int = 64
    epochs: int = 30
    lr: float = 3e-4
    head_lr: Optional[float] = None
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    dropout: float = 0.35
    augmentation: str = "strong"
    channels: Tuple[int, ...] = (32, 64, 128, 256, 384)
    use_batch_norm: bool = True
    patience: int = 8
    freeze_backbone: bool = False


def show_config(config: ExperimentConfig, title: str = "Configuration") -> None:
    frame = pd.DataFrame.from_dict(asdict(config), orient="index", columns=["value"])
    print(title)
    display(frame)


set_seed(SEED)


## Data Loading and Augmentation

For the custom CNN, the best setup uses `strong` augmentation because aircraft variants are fine-grained: color, scale, and crop changes reduce overfitting while preserving the aircraft identity. The ResNet-18 optimized run uses `resnet_strong`, which adds RandAugment when the installed torchvision version supports it.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def randaugment_or_jitter() -> List[Callable]:
    if hasattr(T, "RandAugment"):
        return [T.RandAugment(num_ops=2, magnitude=9)]
    return [T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03)]


def make_transforms(image_size: int, split: str, augmentation: str) -> T.Compose:
    normalize = [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]

    if split == "train":
        if augmentation == "none":
            steps: List[Callable] = [T.Resize((image_size, image_size))]
        elif augmentation == "light":
            resize_size = int(image_size * 1.15)
            steps = [
                T.Resize((resize_size, resize_size)),
                T.RandomCrop(image_size),
                T.RandomHorizontalFlip(p=0.5),
            ]
        elif augmentation == "strong":
            steps = [
                T.RandomResizedCrop(image_size, scale=(0.70, 1.0), ratio=(0.85, 1.15)),
                T.RandomHorizontalFlip(p=0.5),
                T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03),
            ]
        elif augmentation == "resnet_strong":
            steps = [
                T.RandomResizedCrop(image_size, scale=(0.60, 1.0), ratio=(0.80, 1.20)),
                T.RandomHorizontalFlip(p=0.5),
                *randaugment_or_jitter(),
            ]
        else:
            raise ValueError(f"Unknown augmentation policy: {augmentation}")

        steps.extend(normalize)
        if augmentation in {"strong", "resnet_strong"}:
            steps.append(T.RandomErasing(p=0.20, scale=(0.02, 0.12), ratio=(0.3, 3.3)))
        return T.Compose(steps)

    resize_size = int(image_size * 1.15)
    return T.Compose([
        T.Resize((resize_size, resize_size)),
        T.CenterCrop(image_size),
        *normalize,
    ])


def subset_dataset(dataset, max_items: Optional[int] = None, fraction: Optional[float] = None) -> Subset:
    total = len(dataset)
    if max_items is None and fraction is None:
        return dataset
    if fraction is not None:
        max_items = max(1, int(total * fraction))
    max_items = min(total, int(max_items))
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(total, generator=generator)[:max_items].tolist()
    return Subset(dataset, indices)


def dataset_classes(dataset) -> List[str]:
    base = dataset.dataset if isinstance(dataset, Subset) else dataset
    return list(getattr(base, "classes", []))


def make_datasets(config: ExperimentConfig):
    train_dataset = datasets.FGVCAircraft(
        root=DATA_DIR,
        split="train",
        annotation_level="variant",
        download=RUN_DOWNLOAD,
        transform=make_transforms(config.image_size, "train", config.augmentation),
    )
    val_dataset = datasets.FGVCAircraft(
        root=DATA_DIR,
        split="val",
        annotation_level="variant",
        download=RUN_DOWNLOAD,
        transform=make_transforms(config.image_size, "val", config.augmentation),
    )
    test_dataset = datasets.FGVCAircraft(
        root=DATA_DIR,
        split="test",
        annotation_level="variant",
        download=RUN_DOWNLOAD,
        transform=make_transforms(config.image_size, "test", config.augmentation),
    )

    if FAST_DEV_RUN:
        train_dataset = subset_dataset(train_dataset, max_items=512)
        val_dataset = subset_dataset(val_dataset, max_items=256)
        test_dataset = subset_dataset(test_dataset, max_items=256)

    return train_dataset, val_dataset, test_dataset


def make_loaders(config: ExperimentConfig):
    train_dataset, val_dataset, test_dataset = make_datasets(config)
    batch_size = min(config.batch_size, len(train_dataset))

    loaders = {
        "train": DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        ),
        "val": DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        ),
        "test": DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        ),
    }
    class_names = dataset_classes(train_dataset)
    return loaders, class_names


def unnormalize_batch(images: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(IMAGENET_MEAN, device=images.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=images.device).view(1, 3, 1, 1)
    return (images * std + mean).clamp(0, 1)


def show_batch(loader: DataLoader, class_names: Sequence[str], max_images: int = 8) -> None:
    images, labels = next(iter(loader))
    images = unnormalize_batch(images[:max_images]).cpu()
    labels = labels[:max_images]

    cols = min(max_images, 4)
    rows = math.ceil(max_images / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.4 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, image, label in zip(axes, images, labels):
        ax.imshow(image.permute(1, 2, 0))
        ax.set_title(class_names[int(label)], fontsize=9)
        ax.axis("off")
    for ax in axes[len(images):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## Shared Training Utilities

The functions below are used by both the custom CNN and ResNet-18. Each experiment saves three artifacts:

- Best checkpoint: `outputs/checkpoints/<experiment>.pt`
- Per-epoch history: `outputs/histories/<experiment>.csv`
- Test summary: `outputs/<experiment>_summary.json`


In [ ]:
def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    params = model.parameters()
    if trainable_only:
        params = (p for p in params if p.requires_grad)
    return sum(p.numel() for p in params)


def make_optimizer(model: nn.Module, config: ExperimentConfig) -> torch.optim.Optimizer:
    if config.head_lr is not None:
        backbone_params = []
        head_params = []
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if name.startswith("fc.") or name.startswith("classifier."):
                head_params.append(param)
            else:
                backbone_params.append(param)

        groups = []
        if backbone_params:
            groups.append({"params": backbone_params, "lr": config.lr})
        if head_params:
            groups.append({"params": head_params, "lr": config.head_lr})
        return torch.optim.AdamW(groups, weight_decay=config.weight_decay)

    return torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=config.lr,
        weight_decay=config.weight_decay,
    )


def batch_accuracy(logits: torch.Tensor, targets: torch.Tensor) -> int:
    predictions = logits.argmax(dim=1)
    return int((predictions == targets).sum().item())


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
) -> Dict[str, float]:
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    amp_enabled = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled) if training else None

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                logits = model(images)
                loss = criterion(logits, targets)

            if training:
                assert scaler is not None
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(optimizer)
                scaler.update()

        batch_size = targets.size(0)
        total_loss += float(loss.item()) * batch_size
        total_correct += batch_accuracy(logits.detach(), targets)
        total_examples += batch_size

    return {
        "loss": total_loss / max(1, total_examples),
        "acc": total_correct / max(1, total_examples),
    }


@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, criterion: Optional[nn.Module] = None) -> Dict[str, float]:
    model.eval()
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    return run_epoch(model, loader, criterion, optimizer=None)


def fit_model(model: nn.Module, config: ExperimentConfig, loaders: Dict[str, DataLoader]) -> Tuple[nn.Module, pd.DataFrame]:
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, config.epochs))

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0
    epochs_without_improvement = 0
    history: List[Dict[str, float]] = []
    started = time.time()

    print(f"Training {config.name}")
    print(f"Total parameters: {count_parameters(model):,}")
    print(f"Trainable parameters: {count_parameters(model, trainable_only=True):,}")

    for epoch in range(1, config.epochs + 1):
        train_stats = run_epoch(model, loaders["train"], criterion, optimizer=optimizer)
        val_stats = evaluate_model(model, loaders["val"], criterion=criterion)
        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_stats["loss"],
            "train_acc": train_stats["acc"],
            "val_loss": val_stats["loss"],
            "val_acc": val_stats["acc"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d}/{config.epochs} | "
            f"train loss {row['train_loss']:.4f}, acc {row['train_acc']:.3f} | "
            f"val loss {row['val_loss']:.4f}, acc {row['val_acc']:.3f}"
        )

        if val_stats["acc"] > best_val_acc + 1e-4:
            best_val_acc = val_stats["acc"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_state": best_state,
                    "config": asdict(config),
                    "best_val_acc": best_val_acc,
                    "epoch": epoch,
                },
                CHECKPOINT_DIR / f"{config.name}.pt",
            )
        else:
            epochs_without_improvement += 1

        if config.patience and epochs_without_improvement >= config.patience:
            print(f"Early stopping after {epoch} epochs.")
            break

    elapsed_minutes = (time.time() - started) / 60.0
    model.load_state_dict(best_state)
    history_frame = pd.DataFrame(history)
    history_frame["elapsed_minutes_total"] = elapsed_minutes
    history_frame.to_csv(HISTORY_DIR / f"{config.name}.csv", index=False)
    return model, history_frame


def run_experiment(
    config: ExperimentConfig,
    model_builder: Callable[[int, ExperimentConfig], nn.Module],
) -> Dict[str, object]:
    set_seed(SEED)
    loaders, class_names = make_loaders(config)
    model = model_builder(len(class_names), config)
    trained_model, history = fit_model(model, config, loaders)
    test_stats = evaluate_model(trained_model, loaders["test"])

    summary = {
        "experiment": config.name,
        "best_val_acc": float(history["val_acc"].max()),
        "test_loss": float(test_stats["loss"]),
        "test_acc": float(test_stats["acc"]),
        "epochs_ran": int(history["epoch"].max()),
        "trainable_parameters": int(count_parameters(trained_model, trainable_only=True)),
        "total_parameters": int(count_parameters(trained_model)),
        "config": asdict(config),
    }
    with open(OUTPUT_DIR / f"{config.name}_summary.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    print(f"Test accuracy for {config.name}: {test_stats['acc']:.3f}")
    return {
        "config": config,
        "model": trained_model,
        "history": history,
        "test": test_stats,
        "summary": summary,
        "class_names": class_names,
    }


def summarize_results(results: Dict[str, Dict[str, object]]) -> pd.DataFrame:
    rows = []
    for name, result in results.items():
        summary = result["summary"]
        rows.append({
            "experiment": name,
            "best_val_acc": summary["best_val_acc"],
            "test_acc": summary["test_acc"],
            "test_loss": summary["test_loss"],
            "epochs_ran": summary["epochs_ran"],
            "trainable_parameters": summary["trainable_parameters"],
            "augmentation": summary["config"]["augmentation"],
            "lr": summary["config"]["lr"],
            "head_lr": summary["config"]["head_lr"],
            "weight_decay": summary["config"]["weight_decay"],
            "label_smoothing": summary["config"]["label_smoothing"],
            "dropout": summary["config"]["dropout"],
            "freeze_backbone": summary["config"]["freeze_backbone"],
        })
    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame = frame.sort_values("test_acc", ascending=False).reset_index(drop=True)
    return frame


def plot_histories(results: Dict[str, Dict[str, object]], metric: str = "acc") -> None:
    if not results:
        print("No histories available yet. Run the training cells first.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for name, result in results.items():
        history = result["history"]
        axes[0].plot(history["epoch"], history[f"train_{metric}"], label=f"{name} train")
        axes[1].plot(history["epoch"], history[f"val_{metric}"], label=f"{name} val")

    axes[0].set_title(f"Training {metric}")
    axes[1].set_title(f"Validation {metric}")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


# Part 1: Custom CNN From Scratch

The custom model is intentionally built only from PyTorch layers. No `torchvision.models` architecture is used in this part.

The architecture follows a common image-classification pattern: repeated convolutional blocks, spatial downsampling, global average pooling, and a small classifier head. Batch normalization improves optimization stability, dropout reduces overfitting, and data augmentation makes the model less sensitive to aircraft pose, crop, and lighting.


In [ ]:
def activation_layer(name: str = "silu") -> nn.Module:
    if name == "relu":
        return nn.ReLU(inplace=True)
    if name == "gelu":
        return nn.GELU()
    if name == "silu":
        return nn.SiLU(inplace=True)
    raise ValueError(f"Unknown activation: {name}")


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, use_batch_norm: bool = True):
        super().__init__()
        norm1 = nn.BatchNorm2d(out_channels) if use_batch_norm else nn.Identity()
        norm2 = nn.BatchNorm2d(out_channels) if use_batch_norm else nn.Identity()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=not use_batch_norm),
            norm1,
            activation_layer("silu"),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=not use_batch_norm),
            norm2,
            activation_layer("silu"),
            nn.MaxPool2d(kernel_size=2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class AircraftCNN(nn.Module):
    def __init__(
        self,
        num_classes: int,
        channels: Sequence[int] = (32, 64, 128, 256, 384),
        dropout: float = 0.35,
        use_batch_norm: bool = True,
    ):
        super().__init__()
        blocks = []
        in_channels = 3
        for out_channels in channels:
            blocks.append(ConvBlock(in_channels, out_channels, use_batch_norm=use_batch_norm))
            in_channels = out_channels

        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(in_channels, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


def build_custom_cnn(num_classes: int, config: ExperimentConfig) -> nn.Module:
    return AircraftCNN(
        num_classes=num_classes,
        channels=config.channels,
        dropout=config.dropout,
        use_batch_norm=config.use_batch_norm,
    )


## Part 1 Best Model Hyperparameters

This is the main custom-CNN configuration. It uses moderate capacity, batch normalization, dropout, label smoothing, AdamW, cosine learning-rate decay, and strong augmentation. The target is roughly 50 percent test accuracy, but the important part is that the choices are validated by ablations below.


In [ ]:
PART1_BEST_CONFIG = ExperimentConfig(
    name="part1_custom_cnn_best",
    image_size=224,
    batch_size=64,
    epochs=35,
    lr=3e-4,
    head_lr=None,
    weight_decay=1e-4,
    label_smoothing=0.10,
    dropout=0.35,
    augmentation="strong",
    channels=(32, 64, 128, 256, 384),
    use_batch_norm=True,
    patience=8,
    freeze_backbone=False,
)

show_config(PART1_BEST_CONFIG, "Part 1 best custom-CNN hyperparameters")


In [ ]:
part1_results: Dict[str, Dict[str, object]] = {}

if RUN_PART1:
    part1_results[PART1_BEST_CONFIG.name] = run_experiment(PART1_BEST_CONFIG, build_custom_cnn)
    display(summarize_results(part1_results))
else:
    print("RUN_PART1 is False, so the best custom CNN was not trained.")


## Part 1 Ablation: Remove Strong Augmentation

Change: `augmentation="none"` while keeping the architecture and optimizer unchanged.

Expected effect: training accuracy may rise quickly, but validation and test accuracy should drop because the model sees less variation during training and overfits to the training crops.


In [ ]:
PART1_NO_AUG_CONFIG = replace(
    PART1_BEST_CONFIG,
    name="part1_ablation_no_augmentation",
    augmentation="none",
)

show_config(PART1_NO_AUG_CONFIG, "Ablation hyperparameters: no augmentation")


## Part 1 Ablation: Remove Batch Normalization

Change: `use_batch_norm=False` while keeping the rest of the best configuration unchanged.

Expected effect: optimization becomes less stable and usually slower, especially in a deeper custom CNN trained from scratch.


In [ ]:
PART1_NO_BN_CONFIG = replace(
    PART1_BEST_CONFIG,
    name="part1_ablation_no_batch_norm",
    use_batch_norm=False,
)

show_config(PART1_NO_BN_CONFIG, "Ablation hyperparameters: no batch normalization")


## Part 1 Ablation: Remove Dropout

Change: `dropout=0.0` while keeping strong augmentation and batch normalization.

Expected effect: the classifier head has less regularization, so the model is more likely to overfit the fine-grained training set.


In [ ]:
PART1_NO_DROPOUT_CONFIG = replace(
    PART1_BEST_CONFIG,
    name="part1_ablation_no_dropout",
    dropout=0.0,
)

show_config(PART1_NO_DROPOUT_CONFIG, "Ablation hyperparameters: no dropout")


## Part 1 Ablation: Reduce Model Capacity

Change: the channel sequence is reduced from five blocks to four smaller blocks.

Expected effect: the model has fewer filters for fine-grained differences between aircraft variants, so it should underfit compared with the best custom CNN.


In [ ]:
PART1_SMALL_CONFIG = replace(
    PART1_BEST_CONFIG,
    name="part1_ablation_smaller_capacity",
    channels=(24, 48, 96, 192),
    dropout=0.25,
)

show_config(PART1_SMALL_CONFIG, "Ablation hyperparameters: smaller capacity")


In [ ]:
PART1_ABLATION_CONFIGS = [
    PART1_NO_AUG_CONFIG,
    PART1_NO_BN_CONFIG,
    PART1_NO_DROPOUT_CONFIG,
    PART1_SMALL_CONFIG,
]

if RUN_PART1:
    for config in PART1_ABLATION_CONFIGS:
        part1_results[config.name] = run_experiment(config, build_custom_cnn)

    part1_table = summarize_results(part1_results)
    part1_table.to_csv(OUTPUT_DIR / "part1_custom_cnn_ablation_results.csv", index=False)
    display(part1_table)
    plot_histories(part1_results, metric="acc")
else:
    print("RUN_PART1 is False, so custom-CNN ablations were not trained.")


## Part 1 Analysis Guide

Use the generated table and plots to support the written conclusion. A useful discussion compares each ablation against `part1_custom_cnn_best`:

- If removing augmentation lowers test accuracy while training accuracy stays high, the best model benefited from regularization through input diversity.
- If removing batch normalization lowers or destabilizes validation accuracy, the normalization layers made optimization easier.
- If removing dropout increases the train-val gap, the classifier head was overfitting.
- If the smaller network performs worse on both train and validation, the best network needed the extra capacity.


# Part 2: Fine-Tune ResNet-18

This part intentionally uses a PyTorch-provided model, the opposite rule from Part 1. The backbone is initialized with ImageNet-1K V1 weights, then the final fully connected layer is replaced with a 100-class aircraft classifier.

References used for the choices in Part 2B:

- He et al., "Deep Residual Learning for Image Recognition" introduced ResNet: https://arxiv.org/abs/1512.03385
- PyTorch transfer-learning tutorial recommends replacing the final classifier and optionally freezing the backbone: https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
- ImageNet-pretrained torchvision weights use the normalization statistics applied in this notebook: https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html


In [ ]:
def freeze_resnet_backbone(model: nn.Module) -> None:
    for name, parameter in model.named_parameters():
        parameter.requires_grad = name.startswith("fc.")


def unfreeze_model(model: nn.Module) -> None:
    for parameter in model.parameters():
        parameter.requires_grad = True


def build_resnet18_classifier(num_classes: int, config: ExperimentConfig) -> nn.Module:
    try:
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
    except AttributeError:
        model = models.resnet18(pretrained=True)

    in_features = model.fc.in_features
    if config.dropout > 0:
        model.fc = nn.Sequential(
            nn.Dropout(p=config.dropout),
            nn.Linear(in_features, num_classes),
        )
    else:
        model.fc = nn.Linear(in_features, num_classes)

    if config.freeze_backbone:
        freeze_resnet_backbone(model)

    return model


## Part 2A Hyperparameters: ResNet-18 With Part 1 Training Setup

The assignment asks for ResNet-18 to first reuse the training hyperparameters from the best custom CNN. The model architecture changes, but the image size, batch size, epochs, optimizer learning rate, weight decay, label smoothing, and augmentation policy are copied from Part 1.


In [ ]:
RESNET18_2A_CONFIG = replace(
    PART1_BEST_CONFIG,
    name="part2a_resnet18_same_hyperparameters",
    channels=(),
    dropout=0.0,
    freeze_backbone=False,
)

show_config(RESNET18_2A_CONFIG, "Part 2A ResNet-18 hyperparameters")


In [ ]:
part2_results: Dict[str, Dict[str, object]] = {}

if RUN_PART2A:
    part2_results[RESNET18_2A_CONFIG.name] = run_experiment(
        RESNET18_2A_CONFIG,
        build_resnet18_classifier,
    )
    display(summarize_results(part2_results))
else:
    print("RUN_PART2A is False, so the same-hyperparameter ResNet-18 was not trained.")


## Part 2B Hyperparameters: Frozen Backbone Baseline

Change: freeze the ImageNet feature extractor and train only the new classifier head with a larger learning rate. This is fast and tests how much the pretrained representation already transfers to aircraft variants.


In [ ]:
RESNET18_2B_FROZEN_CONFIG = ExperimentConfig(
    name="part2b_resnet18_frozen_backbone",
    image_size=224,
    batch_size=64,
    epochs=10,
    lr=1e-3,
    head_lr=None,
    weight_decay=1e-4,
    label_smoothing=0.05,
    dropout=0.20,
    augmentation="resnet_strong",
    channels=(),
    use_batch_norm=True,
    patience=4,
    freeze_backbone=True,
)

show_config(RESNET18_2B_FROZEN_CONFIG, "Part 2B frozen-backbone hyperparameters")


## Part 2B Hyperparameters: Full Fine-Tuning

Change: unfreeze the whole ResNet-18 and use two learning rates: a smaller one for the pretrained backbone and a larger one for the new classifier head. This usually improves fine-grained classification because the high-level ImageNet features can adapt to aircraft-specific details without being overwritten too aggressively.


In [ ]:
RESNET18_2B_FINETUNE_CONFIG = ExperimentConfig(
    name="part2b_resnet18_full_finetune",
    image_size=224,
    batch_size=64,
    epochs=25,
    lr=1e-4,
    head_lr=5e-4,
    weight_decay=1e-4,
    label_smoothing=0.10,
    dropout=0.20,
    augmentation="resnet_strong",
    channels=(),
    use_batch_norm=True,
    patience=8,
    freeze_backbone=False,
)

show_config(RESNET18_2B_FINETUNE_CONFIG, "Part 2B full-fine-tuning hyperparameters")


In [ ]:
if RUN_PART2B:
    for config in (RESNET18_2B_FROZEN_CONFIG, RESNET18_2B_FINETUNE_CONFIG):
        part2_results[config.name] = run_experiment(config, build_resnet18_classifier)

    part2_table = summarize_results(part2_results)
    part2_table.to_csv(OUTPUT_DIR / "part2_resnet18_results.csv", index=False)
    display(part2_table)
    plot_histories(part2_results, metric="acc")
else:
    print("RUN_PART2B is False, so optimized ResNet-18 experiments were not trained.")


## Final Comparison

After running all experiments, this cell creates one table across both assignment parts. The table is the main evidence for the final written discussion: the custom-CNN ablations should validate the Part 1 design, and the ResNet-18 rows should show the benefit of transfer learning plus tuned hyperparameters.


In [ ]:
all_results: Dict[str, Dict[str, object]] = {}
all_results.update(part1_results)
all_results.update(part2_results)

if all_results:
    final_table = summarize_results(all_results)
    final_table.to_csv(OUTPUT_DIR / "final_model_comparison.csv", index=False)
    display(final_table)
    plot_histories(all_results, metric="acc")
else:
    print("No experiment results yet. Run the Part 1 and Part 2 training cells first.")


## Final Discussion Template

Fill this paragraph after the runs finish, using the generated numbers from `outputs/final_model_comparison.csv`.

The best custom CNN achieved `<test accuracy>` on the test split. The ablation study shows that `<most important component>` was the largest contributor, because removing it changed test accuracy from `<best>` to `<ablation>`. The other ablations show `<short interpretation>`.

For ResNet-18, the same-hyperparameter run achieved `<test accuracy>`, while the tuned fine-tuning setup achieved `<test accuracy>`. The improvement comes from transfer learning, stronger augmentation, and different learning rates for the pretrained backbone and newly initialized classifier. The frozen-backbone comparison reached `<test accuracy>`, which shows whether ImageNet features alone were enough or whether full fine-tuning was necessary for aircraft variants.
